# Check Labels
**Goal of this notebook:** Review labels and get a sample of the rows and manually label, then compare the LLMs labels to mine.

### Review the shifts and no-shifts

In [1]:
import pandas as pd

In [ ]:
labels =pd.read_csv('labeling/llm_labels_simple.csv')

# How many shift, non-shift, and unsure labels do we have?
print(labels["category"].value_counts())

category
no shift    123
shift        24
Name: count, dtype: int64


In [25]:
# how many 3/3 vs 2/3 vs 1/3
print(labels["agreement"].value_counts())

agreement
3/3    140
2/3      7
Name: count, dtype: int64


In [26]:

# Share of rows labeled shift
print(labels["label"].mean())

0.16326530612244897


I'll get 10 random rows to hand label into a csv file. We'll do 5 that are shifts and 5 that are no shift. I realize this means I know 50% will be shift and 50% will be no shift, but we are only sampling a small number to save time and we need to verify both...

In [27]:
# Rows I've already seen in prompt tests. The 3 example rows are not in
# this file because the script skipped them.
SEEN_ROWS = ["row_012", "row_065", "row_072", "row_102", "row_122"]
pool = labels[~labels["row_id"].isin(SEEN_ROWS)]

# Pick 5 random shifts and 5 random no-shifts, from rows I haven't seen.
# random_state=42 makes the "random" pick the same every time I run it.
shifts = pool[pool["label"] == 1].sample(n=5, random_state=42)
no_shifts = pool[pool["label"] == 0].sample(n=5, random_state=42)

# Combine and shuffle so I can't guess the label from the order
# (frac=1 means "all rows, in random order")
check = pd.concat([shifts, no_shifts]).sample(frac=1, random_state=42)

# Save ONLY the row_id and issue, so the model's label stays hidden (blind)
check[["row_id", "issue"]].to_csv(
    "labeling/hand_check_rows.csv", index=False)
print(len(check), "rows to hand-check")

10 rows to hand-check


After this I reviewed each rows packet and hand-labeled the 10 rows in the hand_check_rows.csv file and included a note on why I chose the label, then came back to this notebook to compare the results. Below I'll score the results.

In [28]:
# Excel saved this file in Windows format (cp1252), not UTF-8
mine = pd.read_csv("labeling/hand_labels.csv", encoding="cp1252")
llm = pd.read_csv("labeling/llm_labels_simple.csv")

llm_small = llm[["row_id", "label", "agreement", "reason"]]
compare = mine.merge(llm_small, on="row_id")
compare["match"] = compare["my_label"] == compare["label"]

number_agree = compare["match"].sum()
number_rows = len(compare)
print("Agree on", number_agree, "of", number_rows, "rows")

disagree = compare[compare["match"] == False]

for _, row in disagree.iterrows():
    print("----------")
    print("Row:", row["row_id"], " Issue:", row["issue"])
    print("My label:", row["my_label"], " LLM label:", row["label"],
          " LLM runs agreed:", row["agreement"])
    print("My note:", row["note"])
    print("LLM reason:", row["reason"])

Agree on 9 of 10 rows
----------
Row: row_106  Issue: abortion
My label: 0  LLM label: 1  LLM runs agreed: 3/3
My note: Both weeks include the argument that Trump would leave abortion to the states and that Democrats are wrong about his position. Last week had tweets saying this as well. This week about 10 tweets repeat that same argument, many by sharing Elon Musk’s claim that Harris lied about Trump’s stance. Musk is a new person making an old argument, not a new story.
LLM reason: This week roughly 10-12 tweets center on a new storyline, Elon Musk accusing Kamala Harris of lying about Trump's abortion stance and claims that Trump would not ban abortion nationally, whereas last week's tweets were a general mix of partisan abortion rights and pro-life arguments with no such fact-check storyline.


### Merge Labels
Add the 3 rows that I labeled by hand at the beginning of the labeling process so I could provide  3 examples to the llm prompt.

In [29]:
llm = pd.read_csv("labeling/llm_labels_simple.csv")

# Update the hand checked row that didn't match to my label
llm.loc[llm["row_id"] == "row_106", "label"] = 0
llm.loc[llm["row_id"] == "row_106", "category"] = "no shift"

# Add examples I already labeled
examples = pd.DataFrame({
    "row_id": ["row_015", "row_063", "row_123"],
    "label": [0, 0, 1],
})

llm = pd.concat([llm, examples], ignore_index=True)

# Make sure it looks right, should be 150
print(len(llm), "rows")

# Should be 126:"0", 24:"1"
print(llm["label"].value_counts())


150 rows
label
0    126
1     24
Name: count, dtype: int64


### Save to labels_final.csv

In [30]:
llm.to_csv("labeling/labels_final.csv", index=False)

### Save labels_for_repo (file with only row_id and the label)

In [ ]:
   final = pd.read_csv("labeling/labels_final.csv")
final[["row_id", "label"]].to_csv("../DataFiles/labels_for_repo.csv", index=False)